# Aula 8 – Limpeza de Dados – Parte 2 (Outliers, Categorias, Datas)

## 1. Objetivos da aula
Ao final desta aula, você será capaz de:

* Identificar outliers de forma simples usando estatísticas descritivas e IQR.
* Tomar decisões sobre tratamento de outliers: remover, limitar (clip) ou manter.
* Trabalhar com colunas categóricas usando `astype("category")`.
* Limpar categorias (valores estranhos, padronização).
* Extrair partes de datas com `.dt` (ano, mês, dia da semana etc.).
* Preparar dados temporais para análise (ordenar por data, definir índice temporal).

Esta aula complementa a limpeza da Aula 7, focando em valores extremos, dados categóricos e datas em séries temporais.

## 2. Outliers: o que são e por que importam?
Um outlier é um valor muito distante da maior parte dos dados. Eles podem ser erros de digitação ou casos especiais reais (como Black Friday). Não existe uma solução única; você precisa detectar, avaliar o contexto e decidir como tratá-los.

## 3. Detectando outliers com estatísticas descritivas e IQR
Vamos construir um exemplo com vendas diárias de uma loja:

In [1]:
import pandas as pd

dados = {
    "dia": list(range(1, 16)),
    "faturamento": [3000, 3200, 3100, 2900, 3050,
                    2950, 3100, 3000, 3050, 3100,
                    2900, 2950, 3000, 2900, 50000]  # último dia com valor muito alto
}

df_fat = pd.DataFrame(dados)
df_fat

,dia,faturamento
0,1,3000
1,2,3200
2,3,3100
3,4,2900
4,5,3050
5,6,2950
6,7,3100
7,8,3000
8,9,3050
9,10,3100


### 3.1. Usando describe para ver a distribuição

In [2]:
df_fat["faturamento"].describe()

count       15.000000
mean      6146.666667
std      12131.974675
min       2900.000000
25%       2950.000000
50%       3000.000000
75%       3100.000000
max      50000.000000
Name: faturamento, dtype: float64

### 3.2. Calculando IQR (Intervalo Interquartil)
`IQR = Q3 - Q1`. Valores fora da faixa `[Q1 - 1.5 * IQR, Q3 + 1.5 * IQR]` são candidatos a outliers.

In [3]:
# Selecionando só a coluna faturamento 
f = df_fat["faturamento"]

#Pegando o Quartil 1 e o Quartil 3
q1 = f.quantile(0.25)
q3 = f.quantile(0.75)

#Armazenar no iqr
iqr = q3 - q1

limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

print(f"Limite Inferior: {limite_inferior}")
print(f"Limite Superior: {limite_superior}")

# Mostrando o limite inferior ou o superior

outliers = df_fat[(df_fat["faturamento"] < limite_inferior) | (df_fat["faturamento"] > limite_superior)]

outliers

Limite Inferior: 2725.0
Limite Superior: 3325.0


,dia,faturamento
14,15,50000


## 4. Tratando outliers: remover, limitar ou manter?
Estratégias comuns incluem remover a linha, limitar com `clip` ou manter marcando como atípico.

### 4.1. Removendo outliers

In [ ]:
df_sem_outliers = df_fat[(df_fat["faturamento"] >= limite_inferior) & (df_fat["faturamento"] <= limite_superior)]
df_sem_outliers

### 4.2. Limitando com clip

In [ ]:
df_clipado = df_fat.copy()
df_clipado["faturamento_limpo"] = df_clipado["faturamento"].clip(upper=limite_superior)
df_clipado

## 5. Colunas categóricas: por que usar category?
Usar `astype("category")` economiza memória e facilita operações específicas. Vamos usar um exemplo de forma de pagamento:

In [ ]:
dados_pag = {
    "id_venda": [1, 2, 3, 4, 5, 6],
    "forma_pagamento": ["Crédito", "Débito", "Crédito", "Pix", "Dinheiro", "Crédito"],
    "valor": [100, 50, 200, 80, 40, 150]
}

df_pag = pd.DataFrame(dados_pag)
df_pag["forma_pagamento"] = df_pag["forma_pagamento"].astype("category")
df_pag.dtypes

### 5.2. Vendo categorias e códigos internos

In [ ]:
df_pag["forma_pagamento"].cat.categories

In [ ]:
df_pag["forma_pagamento"].cat.codes

## 6. Limpando e padronizando categorias
É essencial padronizar (ex: minúsculo) e corrigir erros de digitação.

In [ ]:
df_pag["forma_pagamento"] = df_pag["forma_pagamento"].str.lower().astype("category")
df_pag["forma_pagamento"].cat.categories

## 7. Trabalhando com datas: extraindo ano, mês, dia da semana
Após converter para datetime, use o atributo `.dt`.

In [ ]:
datas = pd.date_range(start="2024-10-01", periods=10, freq="D")
dados_vendas = {
    "data": datas,
    "loja": ["Centro", "Centro", "Shopping", "Shopping", "Centro",
             "Centro", "Shopping", "Centro", "Shopping", "Centro"],
    "faturamento": [3000, 3200, 5000, 4500, 2800, 2900, 5200, 3100, 4800, 3500]
}
df_vendas = pd.DataFrame(dados_vendas)
df_vendas["ano"] = df_vendas["data"].dt.year
df_vendas["mes"] = df_vendas["data"].dt.month
df_vendas["dia"] = df_vendas["data"].dt.day
df_vendas

### 7.2. Dia da semana
Se o locale der problema, mapeie manualmente:

In [ ]:
df_vendas["dia_da_semana_num"] = df_vendas["data"].dt.weekday
mapa_dias = {0: "Segunda", 1: "Terça", 2: "Quarta", 3: "Quinta", 4: "Sexta", 5: "Sábado", 6: "Domingo"}
df_vendas["dia_da_semana_nome"] = df_vendas["dia_da_semana_num"].map(mapa_dias)
df_vendas

## 8. Definindo a data como índice e ordenando
Comum em séries temporais para facilitar filtros por intervalo.

In [ ]:
df_vendas_idx = df_vendas.set_index("data").sort_index()
df_vendas_idx["2024-10-03":"2024-10-06"]

## 10. Resumo da aula
Nesta aula, você aprendeu a:

* **Detectar outliers** com `describe`, `quantile` e `IQR`.
* **Tratar outliers** removendo linhas ou limitando valores com `clip`.
* **Trabalhar com colunas categóricas** usando `astype("category")` e padronização.
* **Trabalhar com datas** extraindo partes com `.dt` e definindo índices temporais.

Isso fecha a parte fundamental de limpeza, deixando seus dados prontos para análises confiáveis.